# Real paired multimodal

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
exp = "pbmc_paired"
result_name = "real_paired"
results_path = f"../results/{exp}"

# load results
results_df = pd.read_csv(f"{results_path}/{result_name}_results.csv")

# in the df, replace "_" with " " 
results_df.columns = results_df.columns.str.replace('_', ' ')

# # convert relevant columns to numeric
num_cols = results_df.columns.difference(['model'])
# results_df[num_cols] = results_df[num_cols].apply(pd.to_numeric, errors='coerce')


# take the abs for silhouette domain because it should be low, doesnt matter the sign
results_df['Silhouette domain'] = results_df['Silhouette domain'].abs()
results_df.index = results_df['model']
results_df.drop(columns=['model'], inplace=True)
results_df

average over all datasets

# print it to a table

In [ ]:
# clean summary_df 
summary_df = results_df.groupby(results_df.index).mean()
# remove "RFMALI" 
summary_df = summary_df.loc[summary_df.index != 'RFMALI']

cols_to_keep = ["Accuracy missing", "Alignment score", "FOSCTTM"]
# rename Accuracy missing to Accuracy
summary_df = summary_df[cols_to_keep]
summary_df = summary_df.rename(columns={"Accuracy missing": "Accuracy"})

In [ ]:
# Function to format the top 3 values: bold, underline, italic (adjusted for lower is better columns)
def format_top_3(df):
    formatted_df = df.copy()
    
    # List of columns where lower values are better
    lower_is_better_columns = ['FOSCTTM', 'Silhouette domain']
    
    
    for column in df.columns:  # Start from the first numerical column
        if column in lower_is_better_columns:
            # Find the smallest 3 values (lower is better)
            top_3 = df[column].nsmallest(3).values
        else:
            # Find the largest 3 values (higher is better)
            top_3 = df[column].nlargest(3).values
        
        if len(top_3) >= 3:
            # Apply formatting: bold for best, underline for second, italic for third
            formatted_df.loc[df.index, column] = df[column].apply(
                lambda x: f"\\textbf{{{x:.3f}}}" if x == top_3[0] else 
                            (f"\\underline{{{x:.3f}}}" if x == top_3[1] else 
                            (f"\\textit{{{x:.3f}}}" if x == top_3[2] else f"{x:.3f}"))
            )
    
    return formatted_df

In [ ]:
format_top_3(summary_df)

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(format_top_3(summary_df).to_latex(
        escape=False,
        multirow=True,
        float_format="%.3f"
    )) 

# then copy the output latex table into a .tex file for inclusion in the paper